## 문제 1 : Seq2Seq vs Attention 모델 비교 실험 설계
### 주제 [일본어 → 한국어 번역]

### 데이터 준비 및 전처리 설계

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# ── 디바이스 ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── 기본 하이퍼파라미터 (튜닝 전 baseline) ──
HP = {
    'embed_dim'  : 256,   # embedding 크기
    'hidden_dim' : 512,   # LSTM hidden 크기
    'n_layers'   : 2,     # LSTM 레이어 수
    'dropout'    : 0.5,   # 드롭아웃 비율
    'lr'         : 1e-3,  # Adam 학습률
    'batch_size' : 32,    # 배치 크기
    'clip'       : 1.0,   # gradient clipping
    'epochs'     : 20,    # 학습 epoch
}

# ── Vocab 크기 (STEP 1 설계 기준) ──
SRC_VOCAB = 4000   # 일본어 문자 vocab
TRG_VOCAB = 5000   # 한국어 어절 vocab
PAD_IDX   = 0

### 1. Seq2Seq

In [2]:
# Encoder 정의
class Encoder(nn.Module):
    def __init__(self, vocab, emb, hid, n_layers, drop):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=PAD_IDX)  # 토큰 → 벡터
        self.rnn   = nn.LSTM(emb, hid, n_layers,                    # LSTM 스택
                             dropout=drop, batch_first=True)
        self.drop  = nn.Dropout(drop)

    def forward(self, src):
        # src : (B, src_len)
        emb = self.drop(self.embed(src))          # (B, src_len, emb)
        _, (h, c) = self.rnn(emb)                 # h/c : (n_layers, B, hid)
        return h, c                               # context vector = 마지막 hidden만 반환

In [3]:
# Decoder 정의
class Decoder(nn.Module):
    def __init__(self, vocab, emb, hid, n_layers, drop):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=PAD_IDX)
        self.rnn   = nn.LSTM(emb, hid, n_layers,
                             dropout=drop, batch_first=True)
        self.fc    = nn.Linear(hid, vocab)         # hidden → vocab 확률
        self.drop  = nn.Dropout(drop)

    def forward(self, trg_t, h, c):
        # trg_t : (B,) ← 현재 스텝 토큰 1개
        trg_t = trg_t.unsqueeze(1)                # (B, 1)
        emb   = self.drop(self.embed(trg_t))      # (B, 1, emb)
        out, (h, c) = self.rnn(emb, (h, c))       # context는 h/c로만 전달 ← bottleneck
        pred  = self.fc(out.squeeze(1))            # (B, vocab)
        return pred, h, c

In [4]:
# Seq2Seq 정의
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_ratio=1.0):
        # src : (B, src_len) | trg : (B, trg_len)
        B, T      = trg.shape
        vocab     = self.decoder.fc.out_features
        outputs   = torch.zeros(B, T, vocab).to(device)

        h, c      = self.encoder(src)              # context vector 생성 (1회)
        inp       = trg[:, 0]                      # <SOS> 토큰

        for t in range(1, T):
            pred, h, c  = self.decoder(inp, h, c)  # 이후 h/c만으로 디코딩
            outputs[:, t] = pred
            # teacher forcing : 정답 or 예측 선택
            inp = trg[:, t] if random.random() < teacher_ratio else pred.argmax(1)

        return outputs  # (B, trg_len, vocab)

# ── 인스턴스 생성 ──
enc_v = Encoder(SRC_VOCAB, HP['embed_dim'], HP['hidden_dim'], HP['n_layers'], HP['dropout'])
dec_v = Decoder(TRG_VOCAB, HP['embed_dim'], HP['hidden_dim'], HP['n_layers'], HP['dropout'])
model_vanilla = Seq2Seq(enc_v, dec_v).to(device)
print("Seq2Seq params:", sum(p.numel() for p in model_vanilla.parameters()))

Seq2Seq params: 12225416


### 2. Seq2Seq + Attention

In [5]:
# Bahdanau Attention 정의
class BahdanauAttention(nn.Module):
    def __init__(self, hid):
        super().__init__()
        self.W1    = nn.Linear(hid, hid)           # decoder hidden 투영
        self.W2    = nn.Linear(hid, hid)           # encoder hidden 투영
        self.v     = nn.Linear(hid, 1, bias=False) # 스칼라 에너지 산출

    def forward(self, dec_h, enc_outs):
        # dec_h    : (B, hid)          ← 현재 decoder hidden (최상위 레이어)
        # enc_outs : (B, src_len, hid) ← Encoder 전체 hidden states
        dec_h    = dec_h.unsqueeze(1)              # (B, 1, hid) 브로드캐스팅용
        energy   = self.v(torch.tanh(
                       self.W1(dec_h) + self.W2(enc_outs)))  # (B, src_len, 1)
        attn_w   = F.softmax(energy.squeeze(2), dim=1)       # (B, src_len) 확률화
        context  = torch.bmm(attn_w.unsqueeze(1), enc_outs)  # (B, 1, hid) 가중합
        return context.squeeze(1), attn_w          # context, 시각화용 weight

In [6]:
# Attention Encoder 정의
class AttnEncoder(nn.Module):
    def __init__(self, vocab, emb, hid, n_layers, drop):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=PAD_IDX)
        self.rnn   = nn.LSTM(emb, hid, n_layers,
                             dropout=drop, batch_first=True)
        self.drop  = nn.Dropout(drop)

    def forward(self, src):
        emb            = self.drop(self.embed(src))     # (B, src_len, emb)
        enc_outs, (h, c) = self.rnn(emb)               # enc_outs : (B, src_len, hid) ← 전체 보존
        return enc_outs, h, c                           # Vanilla과 달리 enc_outs도 반환

In [7]:
# Attention Decoder 정의
class AttnDecoder(nn.Module):
    def __init__(self, vocab, emb, hid, n_layers, drop):
        super().__init__()
        self.embed   = nn.Embedding(vocab, emb, padding_idx=PAD_IDX)
        self.attn    = BahdanauAttention(hid)
        # context(hid) + embedding(emb) 를 이어붙여 LSTM 입력
        self.rnn     = nn.LSTM(emb + hid, hid, n_layers,
                               dropout=drop, batch_first=True)
        self.fc      = nn.Linear(hid * 2, vocab)        # hidden + context → vocab
        self.drop    = nn.Dropout(drop)

    def forward(self, trg_t, h, c, enc_outs):
        # trg_t    : (B,)
        # enc_outs : (B, src_len, hid)
        trg_t   = trg_t.unsqueeze(1)                    # (B, 1)
        emb     = self.drop(self.embed(trg_t))          # (B, 1, emb)

        # 매 스텝마다 attention 계산 ← Vanilla과의 핵심 차이
        context, attn_w = self.attn(h[-1], enc_outs)    # h[-1] : 최상위 레이어 hidden
        context = context.unsqueeze(1)                  # (B, 1, hid)

        rnn_in  = torch.cat([emb, context], dim=2)      # (B, 1, emb+hid) concat
        out, (h, c) = self.rnn(rnn_in, (h, c))

        out_cat = torch.cat([out.squeeze(1),
                             context.squeeze(1)], dim=1) # (B, hid*2)
        pred    = self.fc(out_cat)                       # (B, vocab)
        return pred, h, c, attn_w

In [8]:
# Seq2Seq + Attention
class Seq2SeqAttn(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_ratio=1.0):
        B, T        = trg.shape
        vocab       = self.decoder.fc.out_features
        outputs     = torch.zeros(B, T, vocab).to(device)
        attn_maps   = torch.zeros(B, T, src.shape[1]).to(device)  # 시각화용

        enc_outs, h, c = self.encoder(src)             # 전체 hidden 보존
        inp            = trg[:, 0]                     # <SOS>

        for t in range(1, T):
            pred, h, c, attn_w = self.decoder(inp, h, c, enc_outs)  # 매 step attention
            outputs[:, t]   = pred
            attn_maps[:, t] = attn_w
            inp = trg[:, t] if random.random() < teacher_ratio else pred.argmax(1)

        return outputs, attn_maps  # attention map도 함께 반환

# ── 인스턴스 생성 ──
enc_a = AttnEncoder(SRC_VOCAB, HP['embed_dim'], HP['hidden_dim'], HP['n_layers'], HP['dropout'])
dec_a = AttnDecoder(TRG_VOCAB, HP['embed_dim'], HP['hidden_dim'], HP['n_layers'], HP['dropout'])
model_attn = Seq2SeqAttn(enc_a, dec_a).to(device)
print("Seq2Seq+Attention params:", sum(p.numel() for p in model_attn.parameters()))

Seq2Seq+Attention params: 16359816


### 3. 하이퍼파라미터 튜닝

선택 : Random Search

Random Search는 제한된 횟수 내에서 다양한 조합을 탐색해 계산 비용을 줄일 수 있기 때문에,
중요 하이퍼파라미터에 대해 더 다양한 값이 시도되어 적은 실험으로도 좋은 성능을 기대할 수 있음.

In [9]:
import itertools, copy

# ── 탐색 공간 정의 ──
SEARCH_SPACE = {
    'embed_dim'  : [128, 256],           # embedding 크기 후보
    'hidden_dim' : [256, 512],           # hidden 크기 후보
    'lr'         : [1e-3, 5e-4, 1e-4],  # 학습률 후보
    'dropout'    : [0.3, 0.5],           # dropout 후보
    'batch_size' : [32, 64],             # batch 후보
}

import random as rnd

def random_search(space, n_trials=10, seed=42):
    rnd.seed(seed)
    configs = []
    for _ in range(n_trials):
        cfg = {k: rnd.choice(v) for k, v in space.items()}  # 각 HP에서 무작위 선택
        configs.append(cfg)
    return configs

search_configs = random_search(SEARCH_SPACE, n_trials=10)

# ── 탐색 결과 미리보기 ──
print(f"총 {len(search_configs)}개 설정 탐색")
for i, cfg in enumerate(search_configs[:3]):
    print(f"  Config {i+1}: {cfg}")

총 10개 설정 탐색
  Config 1: {'embed_dim': 128, 'hidden_dim': 256, 'lr': 0.0001, 'dropout': 0.5, 'batch_size': 32}
  Config 2: {'embed_dim': 128, 'hidden_dim': 256, 'lr': 0.0001, 'dropout': 0.3, 'batch_size': 32}
  Config 3: {'embed_dim': 256, 'hidden_dim': 256, 'lr': 0.001, 'dropout': 0.3, 'batch_size': 32}


### 4. 모델 학습 및 성능 평가

In [44]:
%pip install "datasets<3.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 21.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [45]:
from datasets import load_dataset

dataset = load_dataset('wmt14', 'de-en')

# wmt14는 약 450만 쌍 → 샘플링 필수 (전체 로드 시 메모리/시간 초과)
train_raw = dataset['train'].shuffle(seed=42).select(range(3000))
valid_raw = dataset['validation'].shuffle(seed=42).select(range(300))
test_raw  = dataset['test'].shuffle(seed=42).select(range(300))

print(f'Train: {len(train_raw)} | Valid: {len(valid_raw)} | Test: {len(test_raw)}')
print('샘플:', train_raw[0]['translation'])

# ── 언어 키 설정 ──
SRC_LANG, TRG_LANG = 'de', 'en'

HfUriError: Invalid HF URI 'hf://datasets/wmt14@b199e406369ec1b7634206d3ded5ba45de2fe696/.huggingface.yaml'. Repository id must be 'namespace/name', got 'wmt14'.

In [ ]:
# ── 토크나이저 + Vocab 클래스 ──
import re

def tokenize_src(text):
    # SRC(독일어): 소문자화 + 구두점 분리 + 공백 분할 (word-level)
    text = text.lower()
    text = re.sub(r'([.,!?;:"])', r' \1 ', text)
    return text.strip().split()

def tokenize_trg(text):
    # TRG(영어): 소문자화 + 구두점 분리 + 공백 분할 (word-level)
    text = text.lower()
    text = re.sub(r'([.,!?;:"])', r' \1 ', text)
    return text.strip().split()

class Vocab:
    def __init__(self, min_freq=2):
        self.w2i = {'<PAD>':0,'<UNK>':1,'<SOS>':2,'<EOS>':3}
        self.i2w = {0:'<PAD>',1:'<UNK>',2:'<SOS>',3:'<EOS>'}
        self.freq = {}
        self.min_freq = min_freq

    def build(self, sentences, tokenize_fn):
        for sent in sentences:
            for tok in tokenize_fn(sent):
                self.freq[tok] = self.freq.get(tok, 0) + 1
        for tok, cnt in sorted(self.freq.items(), key=lambda x: -x[1]):
            if cnt >= self.min_freq and tok not in self.w2i:
                idx = len(self.w2i)
                self.w2i[tok] = idx; self.i2w[idx] = tok

    def encode(self, tokens):
        return [self.w2i.get(t, 1) for t in tokens]

    def decode(self, indices):
        return [self.i2w.get(i, '<UNK>') for i in indices]

    def __len__(self):
        return len(self.w2i)

src_sents = [d['translation'][SRC_LANG] for d in train_raw]
trg_sents = [d['translation'][TRG_LANG] for d in train_raw]

src_vocab = Vocab(min_freq=2); src_vocab.build(src_sents, tokenize_src)
trg_vocab = Vocab(min_freq=2); trg_vocab.build(trg_sents, tokenize_trg)
print(f'SRC(de) vocab: {len(src_vocab)}  TRG(en) vocab: {len(trg_vocab)}')

In [ ]:
# ── PyTorch Dataset + 동적 패딩 DataLoader ──
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 20

class TranslationDataset(Dataset):
    def __init__(self, raw_data):
        self.pairs = []
        for d in raw_data:
            src = d['translation'][SRC_LANG]
            trg = d['translation'][TRG_LANG]
            s = tokenize_src(src); t = tokenize_trg(trg)
            if len(s) > MAX_LEN or len(t) > MAX_LEN:
                continue   # 최대 길이 초과 샘플 제거
            src_ids = src_vocab.encode(s) + [3]        # 단어 → 인덱스 + <EOS>
            trg_ids = [2] + trg_vocab.encode(t) + [3] # <SOS> + 어절 + <EOS>
            self.pairs.append((src_ids, trg_ids))

    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx): return self.pairs[idx]

def collate_fn(batch):
    src_b, trg_b = zip(*batch)
    max_s = max(len(s) for s in src_b)
    max_t = max(len(t) for t in trg_b)
    src_pad = torch.tensor([s + [0]*(max_s-len(s)) for s in src_b])
    trg_pad = torch.tensor([t + [0]*(max_t-len(t)) for t in trg_b])
    return src_pad, trg_pad

train_ds = TranslationDataset(train_raw)
valid_ds = TranslationDataset(valid_raw)
test_ds  = TranslationDataset(test_raw)

train_loader = DataLoader(train_ds, batch_size=HP['batch_size'], shuffle=True,  collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=64,               shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=64,               shuffle=False, collate_fn=collate_fn)
print(f'Train: {len(train_ds)} | Valid: {len(valid_ds)} | Test: {len(test_ds)} (MAX_LEN={MAX_LEN} 필터 후)')

In [ ]:
# ── 실제 vocab 크기로 모델 재생성 ──
SRC_V = len(src_vocab)
TRG_V = len(trg_vocab)
E = HP['embed_dim']; H = HP['hidden_dim']; NL = HP['n_layers']; DR = HP['dropout']

enc_v = Encoder(SRC_V, E, H, NL, DR)
dec_v = Decoder(TRG_V, E, H, NL, DR)
model_vanilla = Seq2Seq(enc_v, dec_v).to(device)

enc_a = AttnEncoder(SRC_V, E, H, NL, DR)
dec_a = AttnDecoder(TRG_V, E, H, NL, DR)
model_attn = Seq2SeqAttn(enc_a, dec_a).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)   # PAD 위치 loss 무시
opt_v = torch.optim.Adam(model_vanilla.parameters(), lr=HP['lr'])
opt_a = torch.optim.Adam(model_attn.parameters(),    lr=HP['lr'])

print(f'Vanilla params : {sum(p.numel() for p in model_vanilla.parameters()):,}')
print(f'Attention params: {sum(p.numel() for p in model_attn.parameters()):,}')

In [ ]:
# ── 학습 / 검증 함수 ──
def train_epoch(model, loader, optimizer, is_attn=False):
    model.train()
    total = 0
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        out = model(src, trg)[0] if is_attn else model(src, trg)   # Attn은 (output, attn_map) 반환
        loss = criterion(out[:,1:].reshape(-1, out.shape[-1]), trg[:,1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), HP['clip'])  # gradient explosion 방지
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def eval_epoch(model, loader, is_attn=False):
    model.eval()
    total = 0
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            out = model(src, trg, teacher_ratio=0.0)[0] if is_attn else model(src, trg, teacher_ratio=0.0)
            total += criterion(out[:,1:].reshape(-1, out.shape[-1]), trg[:,1:].reshape(-1)).item()
    return total / len(loader)

# ── 두 모델 실제 학습 ──
hist_v = {'train':[], 'val':[]}
hist_a = {'train':[], 'val':[]}
best_v = best_a = float('inf')

for epoch in range(1, HP['epochs']+1):
    tv = train_epoch(model_vanilla, train_loader, opt_v, is_attn=False)
    vv = eval_epoch(model_vanilla,  valid_loader,        is_attn=False)
    ta = train_epoch(model_attn,    train_loader, opt_a, is_attn=True)
    va = eval_epoch(model_attn,     valid_loader,        is_attn=True)

    hist_v['train'].append(tv); hist_v['val'].append(vv)
    hist_a['train'].append(ta); hist_a['val'].append(va)

    if vv < best_v: best_v = vv; torch.save(model_vanilla.state_dict(), 'best_vanilla.pt')
    if va < best_a: best_a = va; torch.save(model_attn.state_dict(),    'best_attn.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Ep {epoch:2d} | Vanilla train={tv:.4f} val={vv:.4f} | Attn train={ta:.4f} val={va:.4f}')

print(f'\nBest val loss | Vanilla: {best_v:.4f} | Attention: {best_a:.4f}')

In [ ]:
# ── 실제 학습 결과 Loss 곡선 시각화 ──
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
eps = list(range(1, HP['epochs']+1))

for ax, hist, title in zip(
        axes, [hist_v, hist_a], ['Vanilla Seq2Seq', 'Seq2Seq + Attention']):
    ax.plot(eps, hist['train'], label='Train Loss', marker='o', markersize=3)
    ax.plot(eps, hist['val'],   label='Valid Loss', marker='s', markersize=3, ls='--')
    best_ep = hist['val'].index(min(hist['val'])) + 1
    ax.axvline(x=best_ep, color='red', ls=':', alpha=0.7, label=f'Best epoch={best_ep}')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Train / Validation Loss (실제 학습 결과)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── best weight 로드 + greedy decoding 번역 함수 ──
model_vanilla.load_state_dict(torch.load('best_vanilla.pt', map_location=device))
model_attn.load_state_dict(torch.load('best_attn.pt',    map_location=device))

def translate(model, sentence, max_len=30, is_attn=False):
    model.eval()
    tokens     = tokenize_src(sentence)                          # 소문자화 + word 분리
    src_ids    = src_vocab.encode(tokens) + [3]                  # 인덱스 변환 + <EOS>
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(device)

    with torch.no_grad():
        if is_attn:
            enc_outs, h, c = model.encoder(src_tensor)
        else:
            h, c = model.encoder(src_tensor)

    trg_token = torch.tensor([2]).to(device)                     # 디코딩 시작 = <SOS>
    result = []

    with torch.no_grad():
        for _ in range(max_len):
            if is_attn:
                pred, h, c, _ = model.decoder(trg_token, h, c, enc_outs)
            else:
                pred, h, c = model.decoder(trg_token, h, c)
            top1 = pred.argmax(1)
            if top1.item() == 3: break                           # <EOS> 예측 → 종료
            result.append(top1.item())
            trg_token = top1

    return ' '.join(trg_vocab.decode(result))

print('번역 함수 로드 완료')

In [ ]:
# ── corpus-level BLEU 계산 함수 (smoothing 포함) ──
from collections import Counter
import math

def ngram_count(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def bleu_score(hypotheses, references, max_n=4, smooth=True):
    bp_hyp = bp_ref = 0
    precisions = []
    for n in range(1, max_n+1):
        match_t = cand_t = 0
        for hyp, ref in zip(hypotheses, references):
            h = hyp.split(); r = ref.split()
            h_ng = ngram_count(h,n); r_ng = ngram_count(r,n)
            clipped = {ng: min(c, r_ng[ng]) for ng, c in h_ng.items()}
            match_t += sum(clipped.values())
            cand_t  += max(0, len(h)-n+1)
        p = (match_t+(1 if smooth else 0))/(cand_t+(1 if smooth else 0)) if cand_t else 0
        precisions.append(p)
    for hyp, ref in zip(hypotheses, references):
        bp_hyp += len(hyp.split()); bp_ref += len(ref.split())
    bp = 1.0 if bp_hyp >= bp_ref else math.exp(1 - bp_ref/bp_hyp)   # brevity penalty
    log_avg = sum(math.log(p) if p>0 else float('-inf') for p in precisions) / max_n
    return bp * math.exp(log_avg) * 100

print('BLEU 함수 로드 완료')

In [ ]:
# ── Test set 전체 번역 후 BLEU 계산 ──
refs_c  = []
hyps_v  = []
hyps_a  = []

for d in test_raw:
    ja = d['translation']['ja']; ko = d['translation']['ko']
    refs_c.append(ko)
    hyps_v.append(translate(model_vanilla, ja, is_attn=False))
    hyps_a.append(translate(model_attn,    ja, is_attn=True))

bleu_v = bleu_score(hyps_v, refs_c)
bleu_a = bleu_score(hyps_a, refs_c)

import pandas as pd
df = pd.DataFrame({
    '모델'        : ['Vanilla Seq2Seq', 'Seq2Seq + Attention'],
    'Corpus BLEU' : [round(bleu_v,2),  round(bleu_a,2)],
    'Best Val Loss': [round(best_v,4), round(best_a,4)],
})
print(df.to_string(index=False))
print(f'BLEU 향상: +{round(bleu_a-bleu_v,2)}')

In [ ]:
# ── 3개 테스트 문장 실제 모델 추론 결과 비교 ──
test_sentences_src = [
    'Guten Morgen, wie geht es Ihnen?',
    'Das Modell wird auf seine Uebersetzungsleistung bewertet.',
    'In den letzten Jahren haben sich die Modelle der natuerlichen Sprachverarbeitung durch die Entwicklung grosser Datensaetze und Rechenressourcen erheblich verbessert.',
]
test_targets_trg = [
    'good morning , how are you ?',
    'the model is evaluated on its translation performance .',
    'in recent years , natural language processing models have improved significantly through the development of large datasets and computing resources .',
]

for i, (src, trg) in enumerate(zip(test_sentences_src, test_targets_trg), 1):
    out_v = translate(model_vanilla, src, is_attn=False)
    out_a = translate(model_attn,    src, is_attn=True)
    print(f'{chr(9472)*72}')
    print(f'[{i}] Input  (DE)  : {src}')
    print(f'     Target  (EN)  : {trg}')
    print(f'     Seq2Seq (EN)  : {out_v}')
    print(f'     Attention(EN) : {out_a}')
print(f'{chr(9472)*72}')

In [ ]:
# ── Attention weight 히트맵 : 실제 학습된 모델의 소스-타깃 대응 시각화 ──
import numpy as np
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'

def get_attn_map(model, sentence, max_len=30):
    model.eval()
    tokens = tokenize_src(sentence)
    src_ids = src_vocab.encode(tokens) + [3]
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(device)
    src_disp = tokens + ['<EOS>']

    with torch.no_grad():
        enc_outs, h, c = model.encoder(src_tensor)

    trg_token = torch.tensor([2]).to(device)
    weights = []; trg_disp = []

    with torch.no_grad():
        for _ in range(max_len):
            pred, h, c, attn_w = model.decoder(trg_token, h, c, enc_outs)
            top1 = pred.argmax(1)
            if top1.item() == 3: break
            weights.append(attn_w.squeeze(0).cpu().numpy())   # (src_len,) 저장
            trg_disp.append(trg_vocab.i2w.get(top1.item(), '<UNK>'))
            trg_token = top1

    if not weights:
        return src_disp, ['<UNK>'], np.zeros((1, len(src_disp)))
    mat = np.array(weights)                                    # (trg_len, src_len)
    return src_disp, trg_disp, mat

# 샘플 2번 문장으로 시각화
src_t, trg_t, attn_mat = get_attn_map(model_attn, test_sentences_ja[1])

fig, ax = plt.subplots(figsize=(max(8, len(src_t)), max(3, len(trg_t)//2+2)))
im = ax.imshow(attn_mat, cmap='Blues', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(src_t))); ax.set_xticklabels(src_t, fontsize=9, rotation=45)
ax.set_yticks(range(len(trg_t))); ax.set_yticklabels(trg_t, fontsize=9)
ax.set_xlabel('일본어 Source (문자 단위)', fontsize=11)
ax.set_ylabel('한국어 Target (어절 단위)', fontsize=11)
ax.set_title('Attention Weight Heatmap [실제 학습 모델]', fontsize=12, fontweight='bold')
for i in range(len(trg_t)):
    for j in range(len(src_t)):
        if j < attn_mat.shape[1] and attn_mat[i,j] >= 0.35:
            ax.text(j, i, f'{attn_mat[i,j]:.2f}', ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold')
plt.tight_layout(); plt.show()